<a href="https://colab.research.google.com/github/vamsiram89/-Projects/blob/main/web_scraping_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install requests beautifulsoup4 pandas


In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time

In [36]:
# Function to extract Product Title
def get_title(soup):

    try:
        # Outer Tag Object
        title = soup.find("span", attrs={"id":'productTitle'})

        # Inner NavigatableString Object
        title_value = title.text

        # Title as a string value
        title_string = title_value.strip()

    except AttributeError:
        title_string = ""

    return title_string

# Function to extract Product Price
def get_price(soup):
    try:
        # Attempt 1: Look for any span with class 'a-offscreen' that contains the price
        price_elements = soup.find_all("span", attrs={'class':'a-offscreen'})
        for element in price_elements:
            text = element.get_text(strip=True)
            # A simple heuristic to check if it's a price: starts with a currency symbol or contains digits and a decimal
            if any(char.isdigit() for char in text) and ('₹' in text or '$' in text or '.' in text):
                return text

        # Attempt 2: Fallback to older IDs if 'a-offscreen' doesn't work
        price = soup.find("span", attrs={'id':'priceblock_ourprice'})
        if price:
            return price.get_text(strip=True)

        price = soup.find("span", attrs={'id':'priceblock_dealprice'})
        if price:
            return price.get_text(strip=True)

        return ""
    except Exception:
        return ""

# Function to extract Product Rating
def get_rating(soup):
    try:
        # Most reliable way to get the full rating text
        rating_span = soup.find("span", attrs={'class':'a-icon-alt'})
        if rating_span:
            return rating_span.get_text(strip=True)
        else:
            return ""
    except Exception:
        return ""

# Function to extract Number of User Reviews
def get_review_count(soup):
    try:
        review_count = soup.find("span", attrs={'id':'acrCustomerReviewText'})
        if review_count: return review_count.get_text(strip=True)
        else: return ""

    except AttributeError:
        review_count = ""

    return review_count

# Function to extract Availability Status
def get_availability(soup):
    try:
        available = soup.find("div", attrs={'id':'availability'})
        if available:
            available_span = available.find("span")
            if available_span: return available_span.get_text(strip=True)
        return "Not Available"

    except AttributeError:
        available = "Not Available"

    return available

In [37]:
if __name__ == '__main__':

    # add your user agent
    HEADERS = ({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36', 'Accept-Language': 'en-IN,en;q=0.9'})

    # The webpage URL (using the URL from the kernel state, which is amazon.in for iphone)
    URL = "https://www.amazon.in/s?k=mens+clothes&rh=n%3A1968024031&ref=nb_sb_noss"

    # HTTP Request
    webpage = requests.get(URL, headers=HEADERS)

    # Soup Object containing all data
    soup = BeautifulSoup(webpage.content, "html.parser")

    # Fetch links as List of Tag Objects
    # This part might need adjustment if Amazon changes their search result link classes
    links = soup.find_all("a", attrs={'class':'a-link-normal s-no-outline'}) # or 's-pagination-item' or 'a-link-normal'

    # Store the links
    links_list = []

    # Loop for extracting links from Tag Objects
    for link in links:
            # Ensure to get the href attribute
            href = link.get('href')
            if href and not href.startswith('#'): # Avoid internal anchors
                links_list.append(href)

    d = {"title":[], "price":[], "rating":[], "reviews":[],"availability":[]}

    # Loop for extracting product details from each link
    for link_path in links_list:
        # Construct full URL for product page, handling both relative and absolute paths
        if link_path.startswith("http"):
            full_product_url = link_path
        else:
            full_product_url = "https://www.amazon.in" + link_path

        new_webpage = requests.get(full_product_url, headers=HEADERS)

        new_soup = BeautifulSoup(new_webpage.content, "html.parser")

        # Function calls to display all necessary product information
        d['title'].append(get_title(new_soup))
        d['price'].append(get_price(new_soup))
        d['rating'].append(get_rating(new_soup))
        d['reviews'].append(get_review_count(new_soup))
        d['availability'].append(get_availability(new_soup))


    amazon_df = pd.DataFrame.from_dict(d)
    # amazon_df['title'].replace('', np.nan, inplace=True)
    amazon_df = amazon_df.dropna(subset=['title'])
    amazon_df.to_csv("amazon_lenskart.csv", header=True, index=False)

In [38]:
amazon_df

,title,price,rating,reviews,availability
0,T-Shirt for Men Zip Neck Texture Self Design C...,"₹1,499",4.0 out of 5 stars,(67),In stock
1,KAJARU Men's Polyester Polo T Shirt with Sprea...,₹999,4.2 out of 5 stars,(449),In stock
2,BULLMER Clothing Set with Trendy Shirt & Pants...,"₹2,999",3.8 out of 5 stars,(512),In stock
3,CHKOKKO Men Polyester Solid Quick Dry Half Sle...,"₹1,660",4.0 out of 5 stars,"(4,341)",In stock
4,CHKOKKO Men Polyester Solid Quick Dry Half Sle...,"₹1,660",4.0 out of 5 stars,"(4,341)",In stock
5,Zombom Cotton Polyester Blend Solid Casual Reg...,"₹1,699",3.8 out of 5 stars,"(3,208)",In stock
6,LEOTUDE Men's Half Sleeve Round Neck Cottonble...,"₹1,099",4.0 out of 5 stars,(480),In stock
7,Fitness Mantra® 12 Pairs Sports Ankle Cotton S...,₹199.00,4.0 out of 5 stars,"(4,644)",In stock
8,Mack Jonney Regular Trouser Loose Fit | Sports...,"₹1,400",3.6 out of 5 stars,(940),In stock
9,KOTTY Mens Regular Fit|Classic Design with Sty...,"₹1,999",3.6 out of 5 stars,"(1,236)",In stock
